[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhangdongming0607/ai-memory-course/blob/main/04-memory-storage.ipynb)

> 点击上方按钮，在 Google Colab 中打开本课，可以直接运行代码，无需本地环境。

# 第 4 课：记忆的存储 — 从数据库到向量检索

> 学完这节课，你会：用 JSON 文件和 SQLite 存储记忆、理解 Embedding 和向量相似度、用 ChromaDB 做语义搜索、知道你们系统为什么选了关系型数据库而不是向量数据库。

---

前几课我们学会了调 LLM API、从对话中提取结构化信息。现在问题来了：

**提取出来的记忆，存到哪里？怎么查出来？**

这就是本课的主题。我们从最简单的 JSON 文件开始，一路升级到向量数据库，最后对比你们系统的方案。

## 4.1 最简单的记忆：存到 JSON 文件

最朴素的想法：既然记忆就是一些 key-value 数据，那直接存成 JSON 文件不就行了？

我们来实现一个最简单的 memory store——按联系人 ID 存取记忆。

In [ ]:
import json
import os

class JsonMemoryStore:
    """最简单的记忆存储：一个 JSON 文件"""
    
    def __init__(self, file_path="memories.json"):
        self.file_path = file_path
        # 如果文件已存在就加载，否则从空字典开始
        if os.path.exists(file_path):
            with open(file_path, "r", encoding="utf-8") as f:
                self.data = json.load(f)
        else:
            self.data = {}
    
    def _save(self):
        """把内存中的数据写回文件"""
        with open(self.file_path, "w", encoding="utf-8") as f:
            json.dump(self.data, f, ensure_ascii=False, indent=2)
    
    def add_memory(self, contact_id: str, memory: dict):
        """给某个联系人添加一条记忆"""
        if contact_id not in self.data:
            self.data[contact_id] = {"name": "", "memories": []}
        self.data[contact_id]["memories"].append(memory)
        self._save()
    
    def set_contact_name(self, contact_id: str, name: str):
        """设置联系人名字"""
        if contact_id not in self.data:
            self.data[contact_id] = {"name": name, "memories": []}
        else:
            self.data[contact_id]["name"] = name
        self._save()
    
    def get_memories(self, contact_id: str) -> list:
        """查询某个联系人的所有记忆"""
        if contact_id in self.data:
            return self.data[contact_id]["memories"]
        return []

print("JsonMemoryStore 定义完成")

In [ ]:
# 试试看
store = JsonMemoryStore("demo_memories.json")

# 给联系人添加记忆
store.set_contact_name("contact_001", "小明")
store.add_memory("contact_001", {"fact": "喜欢打羽毛球", "source": "聊天"})
store.add_memory("contact_001", {"fact": "在字节跳动工作", "source": "聊天"})
store.add_memory("contact_001", {"fact": "养了一只猫叫咪咪", "source": "朋友圈"})

store.set_contact_name("contact_002", "小红")
store.add_memory("contact_002", {"fact": "最近在学吉他", "source": "聊天"})
store.add_memory("contact_002", {"fact": "下周要出差去上海", "source": "聊天"})

# 查询
print("小明的记忆：")
for m in store.get_memories("contact_001"):
    print(f"  - {m['fact']}（来源: {m['source']}）")

print("\n小红的记忆：")
for m in store.get_memories("contact_002"):
    print(f"  - {m['fact']}（来源: {m['source']}）")

In [ ]:
# 看看文件长什么样
with open("demo_memories.json", "r", encoding="utf-8") as f:
    print(f.read())

### JSON 文件存储的问题

简单归简单，但问题很多：

| 问题 | 说明 |
|------|------|
| 并发不安全 | 两个进程同时写，数据会丢 |
| 查询效率低 | 所有数据加载到内存，联系人多了就卡 |
| 没有事务 | 写到一半程序崩了，文件就废了 |
| 查询能力弱 | 想查"所有喜欢运动的联系人"？只能遍历 |

所以真实系统都用数据库。

---

## 4.2 用数据库存储（SQLite）

你们系统用的是 **PostgreSQL**（关系型数据库）。这里我们用 **SQLite** 演示——它是 Python 自带的，不用装任何东西。

原理完全一样，只是 SQLite 是单文件数据库，适合学习和小项目。

### 建表

你们系统有 5 张表来存储记忆。我们做一个简化版，用 3 张表：

- **contacts**：联系人基本信息
- **conversation_facts**：从对话中提取的事实
- **memory_items**：汇总后的记忆条目

In [ ]:
import sqlite3
from datetime import datetime

# 创建数据库（文件）和连接
conn = sqlite3.connect("memory.db")
cursor = conn.cursor()

# 建表
cursor.executescript("""
-- 联系人表
CREATE TABLE IF NOT EXISTS contacts (
    id          TEXT PRIMARY KEY,
    name        TEXT NOT NULL,
    relation    TEXT,
    created_at  TEXT DEFAULT (datetime('now'))
);

-- 对话事实表：从每次聊天中提取的原始事实
CREATE TABLE IF NOT EXISTS conversation_facts (
    id          INTEGER PRIMARY KEY AUTOINCREMENT,
    contact_id  TEXT NOT NULL,
    fact        TEXT NOT NULL,
    source      TEXT DEFAULT 'chat',
    extracted_at TEXT DEFAULT (datetime('now')),
    FOREIGN KEY (contact_id) REFERENCES contacts(id)
);

-- 记忆条目表：汇总整理后的记忆
CREATE TABLE IF NOT EXISTS memory_items (
    id          INTEGER PRIMARY KEY AUTOINCREMENT,
    contact_id  TEXT NOT NULL,
    category    TEXT NOT NULL,
    content     TEXT NOT NULL,
    confidence  REAL DEFAULT 1.0,
    updated_at  TEXT DEFAULT (datetime('now')),
    FOREIGN KEY (contact_id) REFERENCES contacts(id)
);
""")

conn.commit()
print("建表完成！")

In [ ]:
# 插入一些示例数据

# 联系人
cursor.executemany(
    "INSERT OR REPLACE INTO contacts (id, name, relation) VALUES (?, ?, ?)",
    [
        ("c001", "小明", "大学同学"),
        ("c002", "小红", "同事"),
        ("c003", "老王", "客户"),
    ]
)

# 对话事实
cursor.executemany(
    "INSERT INTO conversation_facts (contact_id, fact, source) VALUES (?, ?, ?)",
    [
        ("c001", "他说最近在学吉他", "chat"),
        ("c001", "他周末去爬了香山", "chat"),
        ("c001", "他养了一只叫咪咪的猫", "moments"),
        ("c002", "她下周要出差去上海", "chat"),
        ("c002", "她喜欢喝美式咖啡", "chat"),
        ("c003", "他们公司在考虑升级 ERP 系统", "chat"),
    ]
)

# 汇总记忆
cursor.executemany(
    "INSERT INTO memory_items (contact_id, category, content, confidence) VALUES (?, ?, ?, ?)",
    [
        ("c001", "兴趣爱好", "喜欢户外运动，经常爬山", 0.9),
        ("c001", "兴趣爱好", "正在学吉他", 0.8),
        ("c001", "宠物", "养了一只猫叫咪咪", 0.95),
        ("c002", "工作", "近期要出差去上海", 0.85),
        ("c002", "饮食偏好", "喜欢美式咖啡", 0.9),
        ("c003", "商业需求", "在考虑升级 ERP 系统", 0.7),
    ]
)

conn.commit()
print("数据插入完成！")

In [ ]:
# 查询：获取某个联系人的所有记忆
def get_contact_memories(contact_id: str):
    """查询某个联系人的汇总记忆"""
    cursor.execute("""
        SELECT c.name, m.category, m.content, m.confidence
        FROM memory_items m
        JOIN contacts c ON m.contact_id = c.id
        WHERE m.contact_id = ?
        ORDER BY m.category
    """, (contact_id,))
    return cursor.fetchall()

# 查小明的记忆
print("=== 小明的记忆 ===")
for name, category, content, confidence in get_contact_memories("c001"):
    print(f"  [{category}] {content}（置信度: {confidence}）")

print()

# 查所有联系人
print("=== 所有联系人 ===")
cursor.execute("SELECT id, name, relation FROM contacts")
for row in cursor.fetchall():
    print(f"  {row[0]}: {row[1]}（{row[2]}）")

In [ ]:
# SQL 的强大之处：灵活查询

# 查询所有「兴趣爱好」类的记忆
print("=== 所有人的兴趣爱好 ===")
cursor.execute("""
    SELECT c.name, m.content
    FROM memory_items m
    JOIN contacts c ON m.contact_id = c.id
    WHERE m.category = '兴趣爱好'
""")
for name, content in cursor.fetchall():
    print(f"  {name}: {content}")

print()

# 按置信度排序，查高置信度记忆
print("=== 高置信度记忆（>= 0.9）===")
cursor.execute("""
    SELECT c.name, m.category, m.content, m.confidence
    FROM memory_items m
    JOIN contacts c ON m.contact_id = c.id
    WHERE m.confidence >= 0.9
    ORDER BY m.confidence DESC
""")
for name, category, content, conf in cursor.fetchall():
    print(f"  {name} - [{category}] {content}（{conf}）")

### 关系型数据库的优势

| 优势 | 说明 |
|------|------|
| 结构化 | 表结构清晰，字段有类型约束 |
| 可查询 | SQL 可以做各种复杂查询、聚合、排序 |
| 事务安全 | 要么全部写入成功，要么全部回滚 |
| 外键约束 | 保证数据一致性（联系人不存在时不能插记忆） |
| 并发安全 | 数据库自己处理并发锁 |

对于你们的记忆系统来说，数据库完全够用——后面我们会解释为什么。

但有一种需求，SQL 做不了：**语义搜索**。

比如我问"谁喜欢运动"，SQL 只能 `WHERE content LIKE '%运动%'`，精确匹配文字。
如果记忆里写的是"经常爬山""爱打羽毛球"，SQL 就搜不出来了。

这就需要 Embedding 和向量检索。

---

## 4.3 Embedding 是什么

**核心概念：把文本变成一组数字（向量），意思相近的文本，向量也相近。**

想象一个高维空间：
- "他喜欢打羽毛球" → 一个点
- "他爱运动" → 附近的一个点
- "今天天气不错" → 很远的另一个点

这就是 Embedding 做的事情。

In [ ]:
!pip install openai numpy -q

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if api_key:
    print(f"API Key 已加载（前8位: {api_key[:8]}...）")
else:
    print("未找到 OPENAI_API_KEY，请在课程目录创建 .env 文件")

In [ ]:
from openai import OpenAI
import numpy as np

client = OpenAI()

def get_embedding(text: str) -> list[float]:
    """调 OpenAI API 获取文本的 embedding 向量"""
    response = client.embeddings.create(
        model="text-embedding-3-small",  # 便宜的 embedding 模型
        input=text
    )
    return response.data[0].embedding

# 试一下
vec = get_embedding("他喜欢打羽毛球")
print(f"向量维度: {len(vec)}")
print(f"前 10 个数字: {vec[:10]}")
print(f"\n就是这样一串浮点数，总共 {len(vec)} 个。")
print(f"这串数字'编码'了这句话的语义信息。")

In [ ]:
# 余弦相似度：衡量两个向量有多"接近"
# 值在 -1 到 1 之间，越接近 1 越相似

def cosine_similarity(a: list[float], b: list[float]) -> float:
    """计算两个向量的余弦相似度"""
    a = np.array(a)
    b = np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

In [ ]:
# 对几句话生成 embedding，然后算相似度
sentences = [
    "他喜欢打羽毛球",
    "他爱运动",
    "她经常去健身房",
    "今天天气不错",
    "他在字节跳动上班",
]

# 批量获取 embedding
embeddings = {}
for s in sentences:
    embeddings[s] = get_embedding(s)

print("所有句子的 embedding 已生成\n")

# 用第一句话和其他所有句子计算相似度
base = "他喜欢打羽毛球"
print(f"基准句子: '{base}'\n")
print("与其他句子的相似度：")
print("-" * 50)

results = []
for s in sentences:
    if s != base:
        sim = cosine_similarity(embeddings[base], embeddings[s])
        results.append((s, sim))

# 按相似度排序
results.sort(key=lambda x: x[1], reverse=True)
for s, sim in results:
    bar = "#" * int(sim * 40)
    print(f"  {sim:.4f} {bar} '{s}'")

### 看到了吗？

- "他喜欢打羽毛球" 和 "他爱运动" 的相似度最高——虽然没有一个重复的字
- "她经常去健身房" 也不低——都是关于运动的
- "今天天气不错" 和 "他在字节跳动上班" 就离得很远了

**Embedding 理解的是"语义"，不是"文字"。** 这就是它比 SQL `LIKE` 强大的地方。

---

## 4.4 向量数据库

知道了 Embedding 能把文本变成向量、能算相似度，下一个问题是：

**如果我有几万条记忆，怎么快速找到和查询最相似的那几条？**

答案是 **向量数据库** —— 专门存向量、支持"找最相似的"的数据库。

我们用 **ChromaDB**，一个轻量级的向量数据库，本地就能跑，不用装服务器。

In [ ]:
!pip install chromadb -q

In [ ]:
import chromadb

# 创建一个本地的 ChromaDB 实例
chroma_client = chromadb.Client()

# 创建一个 collection（类似数据库里的表）
collection = chroma_client.create_collection(
    name="contact_memories",
    metadata={"hnsw:space": "cosine"}  # 用余弦相似度
)

print("ChromaDB collection 创建完成")

In [ ]:
# 存入一批记忆
memories = [
    {"id": "m1", "text": "小明喜欢打羽毛球，每周打两次", "contact": "小明"},
    {"id": "m2", "text": "小明在字节跳动做前端开发", "contact": "小明"},
    {"id": "m3", "text": "小明养了一只猫叫咪咪", "contact": "小明"},
    {"id": "m4", "text": "小红喜欢喝美式咖啡", "contact": "小红"},
    {"id": "m5", "text": "小红下周要去上海出差", "contact": "小红"},
    {"id": "m6", "text": "小红最近在学瑜伽", "contact": "小红"},
    {"id": "m7", "text": "老王的公司想升级 ERP 系统", "contact": "老王"},
    {"id": "m8", "text": "老王上周请我们吃了日料", "contact": "老王"},
    {"id": "m9", "text": "小明上个月跑了一场半程马拉松", "contact": "小明"},
    {"id": "m10", "text": "小红家里养了一只金毛犬", "contact": "小红"},
]

# 用 OpenAI 生成 embedding 并存入 ChromaDB
for m in memories:
    embedding = get_embedding(m["text"])
    collection.add(
        ids=[m["id"]],
        documents=[m["text"]],
        embeddings=[embedding],
        metadatas=[{"contact": m["contact"]}]
    )

print(f"已存入 {collection.count()} 条记忆")

In [ ]:
# 语义搜索！
def search_memories(query: str, top_k: int = 3):
    """用自然语言搜索记忆"""
    query_embedding = get_embedding(query)
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )
    return results

# 试几个查询
queries = [
    "谁喜欢运动",
    "谁有宠物",
    "工作相关的信息",
]

for q in queries:
    print(f"\n查询: '{q}'")
    print("-" * 40)
    results = search_memories(q)
    for i, (doc, distance, metadata) in enumerate(
        zip(results["documents"][0], results["distances"][0], results["metadatas"][0])
    ):
        similarity = 1 - distance  # ChromaDB 返回的是距离，越小越相似
        print(f"  {i+1}. [{metadata['contact']}] {doc}（相似度: {similarity:.3f}）")

### SQL 精确匹配 vs 向量语义搜索

看看同一个查询，两种方式的差别：

In [ ]:
# 对比：SQL 精确匹配 vs 向量语义搜索

search_term = "运动"

# === SQL 方式：精确匹配 ===
print("=== SQL LIKE 匹配 ===")
cursor.execute("""
    SELECT c.name, m.content 
    FROM memory_items m
    JOIN contacts c ON m.contact_id = c.id
    WHERE m.content LIKE ?
""", (f"%{search_term}%",))

sql_results = cursor.fetchall()
if sql_results:
    for name, content in sql_results:
        print(f"  {name}: {content}")
else:
    print("  没有找到包含'运动'这两个字的记忆")
    # 看看数据库里有什么
    cursor.execute("SELECT c.name, m.content FROM memory_items m JOIN contacts c ON m.contact_id = c.id")
    print("\n  数据库里实际存的：")
    for name, content in cursor.fetchall():
        print(f"    {name}: {content}")

print()

# === 向量方式：语义搜索 ===
print("=== 向量语义搜索 ===")
results = search_memories("运动", top_k=3)
for doc, metadata in zip(results["documents"][0], results["metadatas"][0]):
    print(f"  [{metadata['contact']}] {doc}")

print()
print("区别在于：")
print("  SQL 只能找到包含'运动'这两个字的记录")
print("  向量搜索能找到'打羽毛球''跑马拉松''学瑜伽'——它理解这些都和运动有关")

---

## 4.5 你们系统的选择

现在你知道了两种存储方案：关系型数据库（SQL）和向量数据库。

**你们系统选的是关系型数据库（PostgreSQL），不用向量检索。**

这不是因为向量数据库不好，而是因为你们的场景不需要。

### 为什么？

你们的记忆是 **"联系人维度"** 的：
- 用户打开和小明的聊天 → 系统查 `WHERE contact_id = '小明'` → 返回小明的所有记忆
- 用户要回复小红的消息 → 系统查 `WHERE contact_id = '小红'` → 返回小红的所有记忆

**永远知道该查谁的记忆，不需要语义搜索。**

### Mem0 为什么用向量检索？

Mem0 是一个通用的记忆框架，它面向的场景更广：
- 用户可能说："上次那个喜欢运动的朋友叫什么来着？"
- 系统不知道该查谁 → 只能用语义搜索，从所有记忆中找最相关的

In [ ]:
# 用代码对比两种方案

# === 你们系统的做法 ===
def your_system_recall(contact_id: str):
    """你们系统：按联系人 ID 直接查"""
    cursor.execute("""
        SELECT category, content FROM memory_items
        WHERE contact_id = ?
        ORDER BY category
    """, (contact_id,))
    return cursor.fetchall()

# === Mem0 式的做法 ===
def mem0_style_recall(query: str, top_k: int = 5):
    """Mem0 风格：语义搜索所有记忆"""
    query_embedding = get_embedding(query)
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )
    return list(zip(results["documents"][0], results["metadatas"][0]))

# 场景：用户正在和小明聊天，需要查小明的记忆
print("=== 场景：回复小明的消息 ===")
print()

print("你们系统的做法（按 contact_id 查）：")
for category, content in your_system_recall("c001"):
    print(f"  [{category}] {content}")

print()
print("Mem0 式做法（语义搜索'小明相关'）：")
for doc, meta in mem0_style_recall("小明的个人信息和爱好"):
    print(f"  [{meta['contact']}] {doc}")

In [ ]:
# 对比表格
comparison = [
    ["维度", "你们系统（关系型数据库）", "Mem0（向量数据库）"],
    ["存储方式", "PostgreSQL 表", "向量数据库 + 关系型数据库"],
    ["查询方式", "按联系人 ID 精确查询", "语义相似度搜索"],
    ["查询场景", "已知要查谁的记忆", "不确定该查谁的记忆"],
    ["查询速度", "极快（索引命中）", "较快（向量索引，但比不上主键查询）"],
    ["适用场景", "联系人维度的记忆系统", "通用记忆 / 知识库"],
    ["实现复杂度", "低（标准 SQL）", "中（需要 embedding + 向量数据库）"],
    ["成本", "只有数据库成本", "额外的 embedding API 调用费用"],
    ["典型查询", "SELECT * WHERE contact_id = ?", "给我和'运动'最相关的记忆"],
]

# 打印对比表
col_widths = [max(len(row[i]) for row in comparison) for i in range(3)]
# 中文字符宽度修正
def display_width(s):
    width = 0
    for c in s:
        width += 2 if ord(c) > 127 else 1
    return width

def pad(s, target_width):
    return s + " " * (target_width - display_width(s))

target_widths = [max(display_width(row[i]) for row in comparison) for i in range(3)]

for idx, row in enumerate(comparison):
    line = " | ".join(pad(row[i], target_widths[i]) for i in range(3))
    print(line)
    if idx == 0:
        print("-" * len(line.encode('utf-8')))

### 选择总结

**没有"更好"的方案，只有"更合适"的方案。**

- 你们做的是微信场景的记忆系统，每条记忆都关联到具体的联系人 → **关系型数据库足够了**
- 如果你要做一个通用的 AI 助手记忆系统，需要从海量记忆中找最相关的 → **向量数据库更合适**
- 实际上两者也可以结合：先按联系人 ID 缩小范围，再在小范围内做语义搜索

---

## 4.6 本课小结

| 概念 | 你学到了什么 |
|------|----------|
| JSON 存储 | 最简单但不可靠，适合 demo |
| SQLite/SQL | 结构化、可查询、事务安全，适合生产 |
| Embedding | 把文本变成向量，语义相近的文本向量也相近 |
| 余弦相似度 | 衡量两个向量有多"接近"，范围 -1 到 1 |
| 向量数据库 | 专门存向量、做"最相似"搜索的数据库 |
| SQL vs 向量搜索 | SQL 精确匹配文字，向量搜索理解语义 |
| 你们的选择 | 按联系人 ID 查就行，不需要向量检索 |

### 关键认识

**存储方案取决于你的查询模式：**
- 知道"查谁的" → 关系型数据库（按 ID 查）
- 不知道"查谁的"、需要"找最相关的" → 向量数据库（语义搜索）

### 下节预告

下一课我们把前面学的串起来，看看一条记忆从"聊天消息"到"被 AI 使用"的完整旅程——记忆系统的完整 Pipeline。

In [ ]:
# 清理临时文件
import os

conn.close()

for f in ["demo_memories.json", "memory.db"]:
    if os.path.exists(f):
        os.remove(f)
        print(f"已清理: {f}")

print("\n清理完成！")